<a href="https://colab.research.google.com/github/mloulaps-rgb/---/blob/main/HCI_%D0%9B%D0%A05_%D0%90%D0%B2%D1%82%D0%BE%D0%BA%D0%BE%D1%80%D0%B5%D0%BA%D1%86%D1%96%D1%8F_%D1%82%D0%B5%D0%BA%D1%81%D1%82%D1%83_%D0%92%D0%B5%D0%BB%D0%B8%D1%87%D0%BA%D0%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Лабораторна робота №5. Автокорекція тексту з мінімальною відстанню редагування.
# Крок 1. Обробка корпусу, побудова словника частотності та ймовірностей слів.
import re
from collections import Counter
import numpy as np
import pandas as pd

# Корпус тексту - твори Шекспіра
!wget -q -O shakespeare.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

def process_data(file_name):
    """Зчитує файл і повертає список слів у нижньому регістрі."""
    with open(file_name, "r") as f:
        text = f.read().lower()
    return re.findall(r'\w+', text)

def get_count(word_l):
    """Створює словник частотності слів."""
    word_count_dict = {}
    for w in word_l:
        word_count_dict[w] = word_count_dict.get(w, 0) + 1
    return word_count_dict

def get_probs(word_count_dict):
    """Обчислює ймовірності слів на основі частот."""
    total = sum(word_count_dict.values())
    return {word: count / total for word, count in word_count_dict.items()}

word_l = process_data('shakespeare.txt')
vocab = set(word_l)
word_count_dict = get_count(word_l)
probs = get_probs(word_count_dict)

print("Усього слів у корпусі:", len(word_l))
print("Унікальних слів (розмір словника):", len(vocab))
print("Найчастіші слова:", Counter(word_count_dict).most_common(5))
print("P('the') =", round(probs['the'], 5), "  P('love') =", round(probs['love'], 6))

Усього слів у корпусі: 208530
Унікальних слів (розмір словника): 11456
Найчастіші слова: [('the', 6287), ('and', 5690), ('i', 5111), ('to', 4934), ('of', 3760)]
P('the') = 0.03015   P('love') = 0.002072


In [2]:
# Крок 2. Функції маніпуляції рядками: видалення, перестановка, заміна, вставка.
def delete_letter(word):
    """Видалення однієї літери."""
    split_l = [(word[:i], word[i:]) for i in range(len(word))]
    return [L + R[1:] for L, R in split_l if R]

def switch_letter(word):
    """Перестановка двох сусідніх літер."""
    split_l = [(word[:i], word[i:]) for i in range(len(word))]
    return [L + R[1] + R[0] + R[2:] for L, R in split_l if len(R) > 1]

def replace_letter(word):
    """Заміна однієї літери іншою."""
    letters = 'abcdefghijklmnopqrstuvwxyz'
    split_l = [(word[:i], word[i:]) for i in range(len(word))]
    return [L + c + R[1:] for L, R in split_l if R for c in letters if L + c + R[1:] != word]

def insert_letter(word):
    """Вставка однієї літери."""
    letters = 'abcdefghijklmnopqrstuvwxyz'
    split_l = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    return [L + c + R for L, R in split_l for c in letters]

print("delete_letter('cans')   ->", delete_letter('cans'))
print("switch_letter('eta')    ->", switch_letter('eta'))
print("replace_letter('can')[:6] ->", replace_letter('can')[:6], "...")
print("insert_letter('at')[:6]   ->", insert_letter('at')[:6], "...")
print("Кількість варіантів insert для 'at':", len(insert_letter('at')))

delete_letter('cans')   -> ['ans', 'cns', 'cas', 'can']
switch_letter('eta')    -> ['tea', 'eat']
replace_letter('can')[:6] -> ['aan', 'ban', 'dan', 'ean', 'fan', 'gan'] ...
insert_letter('at')[:6]   -> ['aat', 'bat', 'cat', 'dat', 'eat', 'fat'] ...
Кількість варіантів insert для 'at': 78


In [3]:
# Крок 4. Пошук слів на відстані одного та двох редагувань.
def edit_one_letter(word, allow_switches=True):
    """Усі слова на відстані одного редагування."""
    edit_set = set()
    edit_set.update(delete_letter(word))
    edit_set.update(replace_letter(word))
    edit_set.update(insert_letter(word))
    if allow_switches:
        edit_set.update(switch_letter(word))
    return edit_set

def edit_two_letters(word, allow_switches=True):
    """Усі слова на відстані двох редагувань."""
    edit_set = set()
    for w in edit_one_letter(word, allow_switches):
        edit_set.update(edit_one_letter(w, allow_switches))
    return edit_set

print("Кількість кандидатів на відстані 1 редагування від 'at':", len(edit_one_letter('at')))
print("Кількість кандидатів на відстані 2 редагувань від 'at':", len(edit_two_letters('at')))

Кількість кандидатів на відстані 1 редагування від 'at': 129
Кількість кандидатів на відстані 2 редагувань від 'at': 7154


In [4]:
# Кроки 3,6. Обчислення ймовірностей кандидатів та система автокорекції.
def get_corrections(word, probs, vocab, n=2, verbose=False):
    """Повертає n найімовірніших корекцій слова."""
    if word in vocab:
        suggestions = [word]
    else:
        edit_one_set = edit_one_letter(word) & vocab
        if edit_one_set:
            suggestions = list(edit_one_set)
        else:
            edit_two_set = edit_two_letters(word) & vocab
            suggestions = list(edit_two_set) if edit_two_set else [word]

    best_words = {w: probs.get(w, 0) for w in suggestions}
    n_best = sorted(best_words.items(), key=lambda x: x[1], reverse=True)[:n]
    if verbose:
        print(f"слово = {word}, кандидатів = {len(suggestions)}")
    return n_best

test_words = ['hllo', 'mony', 'comming', 'happpy', 'dnace', 'thw']
for word in test_words:
    corrections = get_corrections(word, probs, vocab, 3)
    print(f"Корекції для '{word}':")
    for i, (c, p) in enumerate(corrections, 1):
        print(f"  {i}. {c}  (ймовірність: {p:.6f})")
    print()

Корекції для 'hllo':
  1. all  (ймовірність: 0.004364)
  2. ll  (ймовірність: 0.002781)
  3. hold  (ймовірність: 0.000456)

Корекції для 'mony':
  1. many  (ймовірність: 0.000681)
  2. money  (ймовірність: 0.000101)
  3. cony  (ймовірність: 0.000014)

Корекції для 'comming':
  1. coming  (ймовірність: 0.000173)

Корекції для 'happpy':
  1. happy  (ймовірність: 0.000326)

Корекції для 'dnace':
  1. dance  (ймовірність: 0.000086)

Корекції для 'thw':
  1. the  (ймовірність: 0.030149)
  2. thy  (ймовірність: 0.005078)
  3. thaw  (ймовірність: 0.000010)



In [5]:
# Крок 5. Алгоритм мінімальної відстані редагування (динамічне програмування).
def min_edit_distance(source, target, ins_cost=1, del_cost=1, rep_cost=2):
    """Мінімальна відстань редагування між двома рядками."""
    m, n = len(source), len(target)
    D = np.zeros((m + 1, n + 1), dtype=int)
    for row in range(m + 1):
        D[row, 0] = row * del_cost
    for col in range(n + 1):
        D[0, col] = col * ins_cost
    for row in range(1, m + 1):
        for col in range(1, n + 1):
            r_cost = 0 if source[row - 1] == target[col - 1] else rep_cost
            D[row, col] = min(D[row - 1, col] + del_cost,
                              D[row, col - 1] + ins_cost,
                              D[row - 1, col - 1] + r_cost)
    return D, D[m, n]

for source, target in [('play', 'stay'), ('sunday', 'saturday'), ('intention', 'execution')]:
    D, med = min_edit_distance(source, target)
    print(f"Мінімальна відстань редагування '{source}' -> '{target}': {med}")
    print(pd.DataFrame(D, index=['#'] + list(source), columns=['#'] + list(target)))
    print()

Мінімальна відстань редагування 'play' -> 'stay': 4
   #  s  t  a  y
#  0  1  2  3  4
p  1  2  3  4  5
l  2  3  4  5  6
a  3  4  5  4  5
y  4  5  6  5  4

Мінімальна відстань редагування 'sunday' -> 'saturday': 4
   #  s  a  t  u  r  d  a  y
#  0  1  2  3  4  5  6  7  8
s  1  0  1  2  3  4  5  6  7
u  2  1  2  3  2  3  4  5  6
n  3  2  3  4  3  4  5  6  7
d  4  3  4  5  4  5  4  5  6
a  5  4  3  4  5  6  5  4  5
y  6  5  4  5  6  7  6  5  4

Мінімальна відстань редагування 'intention' -> 'execution': 8
   #  e  x   e   c   u   t   i   o   n
#  0  1  2   3   4   5   6   7   8   9
i  1  2  3   4   5   6   7   6   7   8
n  2  3  4   5   6   7   8   7   8   7
t  3  4  5   6   7   8   7   8   9   8
e  4  3  4   5   6   7   8   9  10   9
n  5  4  5   6   7   8   9  10  11  10
t  6  5  6   7   8   9   8   9  10  11
i  7  6  7   8   9  10   9   8   9  10
o  8  7  8   9  10  11  10   9   8   9
n  9  8  9  10  11  12  11  10   9   8



In [6]:
# Крок 7. Оцінка точності та ефективності системи автокорекції.
import random, time
random.seed(1)

# Беремо найчастіші слова (довжиною >= 4) і штучно вносимо по одній помилці,
# потім перевіряємо, чи відновлює їх автокоректор (топ-1 корекція).
freq_words = [w for w, _ in sorted(word_count_dict.items(), key=lambda x: -x[1])
              if len(w) >= 4][:300]

def make_typo(w):
    op = random.choice(['delete', 'swap', 'replace'])
    i = random.randrange(len(w))
    if op == 'delete':
        return w[:i] + w[i + 1:]
    if op == 'swap' and i < len(w) - 1:
        return w[:i] + w[i + 1] + w[i] + w[i + 2:]
    j = random.randrange(26)
    return w[:i] + 'abcdefghijklmnopqrstuvwxyz'[j] + w[i + 1:]

correct = 0
t0 = time.time()
for w in freq_words:
    typo = make_typo(w)
    sugg = get_corrections(typo, probs, vocab, n=1)
    if sugg and sugg[0][0] == w:
        correct += 1
elapsed = time.time() - t0

accuracy = correct / len(freq_words)
print(f"Перевірено слів із помилками: {len(freq_words)}")
print(f"Правильно виправлено (топ-1): {correct}")
print(f"Точність автокорекції: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Загальний час: {elapsed:.3f} с  ({elapsed/len(freq_words)*1000:.2f} мс на слово)")

Перевірено слів із помилками: 300
Правильно виправлено (топ-1): 240
Точність автокорекції: 0.8000 (80.00%)
Загальний час: 0.028 с  (0.09 мс на слово)
